<a href="https://colab.research.google.com/github/yanheng0/SBS-BioHackathon/blob/main/BioHackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BioHackathon


## The Challenge


PCOS represents a microcosm of a larger problem in women's health. Medical conditions that are common, heterogeneous, and poorly understood in medical practice, lead to delayed or missed diagnoses, with disproportionate impact on marginalized populations. Yet early diagnosis matters. Interventions for metabolic complications are most effective when implemented early, and patients deserve timely answers about their health.


The challenge is to develop a feasible system or method that helps medical professionals improve diagnostic accuracy for women's health conditions. The solution should:

Address the diagnostic challenges outlined above such as helping clinicians synthesize complex symptom patterns, identify patients at highest risk of missed diagnosis, differentiate PCOS from conditions with overlapping presentations, or otherwise improve the diagnostic pathway.

Be grounded in available data (such as the datasets provided or other women's health data) and feasible for real-world implementation.

Consider how the solution might address disparities in diagnosis across different populations.

Articulate both the potential impact on patient outcomes and the practical steps toward implementation.


Interpretation is open. The solution need not be limited to machine learning models or mobile apps. It could be a clinical decision support tool, a screening algorithm, a reimagining of diagnostic criteria, a system for identifying at-risk populations, a patient-facing tool for better symptom communication, or any other approach that uses data to address the problem. The key is to demonstrate how the method could reduce diagnostic delays and misdiagnosis, ultimately improving outcomes for women.


# Datasets

## Onedrive:
https://entuedu-my.sharepoint.com/:f:/g/personal/echia013_e_ntu_edu_sg/IgBzV0pH0cC3SaJXYAbofSl_AYFyPgCjaxK0x07e-znUyxY

### PCOS:
https://entuedu-my.sharepoint.com/:x:/r/personal/echia013_e_ntu_edu_sg/_layouts/15/Doc.aspx?sourcedoc=%7B76F2A272-4664-4D93-B370-95AFCBFF889F%7D&file=(Main_Dataset)_PCOS_data_without_infertility.xlsx&action=default&mobileredirect=true

### Endometriosis:
https://entuedu-my.sharepoint.com/:x:/r/personal/echia013_e_ntu_edu_sg/_layouts/15/Doc.aspx?sourcedoc=%7B0A4BFC91-0751-47B1-89C3-C22135369F26%7D&file=(Supplementary_Dataset)_structured_endometriosis_data.csv&action=default&mobileredirect=true

# 1. Code


In [ ]:
!git clone https://github.com/yanheng0/SBS-BioHackathon.git

%cd SBS-BioHackathon

Cloning into 'SBS-BioHackathon'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), 258.40 KiB | 9.94 MiB/s, done.
/content/SBS-BioHackathon/pcos_navigator/SBS-BioHackathon


In [ ]:
!unzip -q pcos_navigator_1.zip

%cd pcos_navigator

!pip install -r requirements.txt

/content/SBS-BioHackathon/pcos_navigator/SBS-BioHackathon/pcos_navigator


# 2. Model Training & Evaluation

In [ ]:
!python train.py --pcos_path "data/PCOS_data_without_infertility.csv" --endo_path "data/_structured_endometriosis_data.csv"

Loading datasets...
  PCOS: 541 patients (177 PCOS, 364 controls)
  Endo: 10000 patients

── Tier 1 — Logistic Regression ──
  roc_auc     : 0.884 ± 0.018
  recall      : 0.820 ± 0.027
  precision   : 0.713 ± 0.036
  f1          : 0.762 ± 0.014
  Saved tier1_lr.pkl  (features: 14)

── Tier 2 — Random Forest ──
  roc_auc     : 0.960 ± 0.009
  recall      : 0.842 ± 0.085
  precision   : 0.866 ± 0.029
  f1          : 0.852 ± 0.054
  Saved tier2_rf.pkl  (features: 19)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:31:18] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:31:19] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:2

# 3. Streamlit

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os
import json

import sys
sys.path.append(os.path.dirname(os.path.dirname(__file__)))
from utils.rotterdam import apply_rotterdam, RotterdamResult

# ── Page config ──────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="PCOS Phenotype Navigator",
    page_icon="🔬",
    layout="wide",
    initial_sidebar_state="expanded",
)

MODELS_DIR = os.path.join(os.path.dirname(__file__), 'models')


# ── Load models (cached) ──────────────────────────────────────────────────────
@st.cache_resource
def load_models():
    models = {}
    for name in ['tier1_lr', 'tier2_rf', 'diff_xgb']:
        path = os.path.join(MODELS_DIR, f'{name}.pkl')
        if os.path.exists(path):
            models[name] = joblib.load(path)
    for name in ['tier1_features', 'tier2_features', 'diff_features']:
        path = os.path.join(MODELS_DIR, f'{name}.pkl')
        if os.path.exists(path):
            models[name] = joblib.load(path)
    metrics_path = os.path.join(MODELS_DIR, 'metrics.json')
    if os.path.exists(metrics_path):
        with open(metrics_path) as f:
            models['metrics'] = json.load(f)
    return models


# ── Sidebar ───────────────────────────────────────────────────────────────────
def render_sidebar():
    st.sidebar.title("PCOS Phenotype Navigator")
    st.sidebar.caption("Clinical decision support — for professional use only")

    setting = st.sidebar.selectbox(
        "Clinical setting",
        ["Primary care / telehealth (no ultrasound)",
         "Specialist clinic (ultrasound available)"],
    )
    tier = 1 if "telehealth" in setting else 2

    st.sidebar.divider()
    ethnicity = st.sidebar.selectbox(
        "Patient ethnicity (for hirsutism threshold)",
        ["Default (≥4)", "East Asian (≥4)", "Caucasian / Black (≥6)"],
    )
    ethnicity_map = {
        "Default (≥4)": "default",
        "East Asian (≥4)": "east_asian",
        "Caucasian / Black (≥6)": "caucasian",
    }

    guideline = st.sidebar.radio(
        "Follicle threshold guideline",
        ["2023 (≥20 follicles/ovary) — recommended",
         "Pre-2023 (≥10 follicles/ovary)"],
    )
    use_2023 = "2023" in guideline

    st.sidebar.divider()
    st.sidebar.caption(
        "⚠️ Dataset note: Follicle count ≥20 threshold is met by only 3.4% "
        "of PCOS patients in the training data (collected under pre-2023 criteria). "
        "The model's ML output is not affected, but the Rotterdam mapping will "
        "differ substantially between guideline versions."
    )

    return tier, ethnicity_map[ethnicity], use_2023


# ── Patient input form ────────────────────────────────────────────────────────
def render_input_form(tier: int):
    st.header("Patient data entry")

    col1, col2, col3 = st.columns(3)

    with col1:
        st.subheader("Demographics & vitals")
        age = st.number_input("Age (years)", 18, 50, 28)
        weight = st.number_input("Weight (kg)", 30.0, 150.0, 65.0, step=0.5)
        height = st.number_input("Height (cm)", 140.0, 200.0, 162.0, step=0.5)
        bmi = round(weight / (height / 100) ** 2, 1)
        st.metric("BMI (calculated)", bmi)

    with col2:
        st.subheader("Hormonal profile")
        lh = st.number_input("LH (mIU/mL)", 0.0, 100.0, 5.0, step=0.1,
                             help="Note: values >100 are capped in this model due to data quality issues in training set")
        fsh = st.number_input("FSH (mIU/mL)", 0.0, 30.0, 6.0, step=0.1)
        fsh_lh = round(fsh / lh, 2) if lh > 0 else 0.0
        st.metric("FSH/LH ratio (calculated)", fsh_lh,
                  delta="⚠️ Inverted" if fsh_lh < 1 else "Normal",
                  delta_color="inverse")
        amh = st.number_input("AMH (ng/mL)", 0.0, 20.0, 3.0, step=0.1)
        tsh = st.number_input("TSH (mIU/L)", 0.0, 10.0, 2.0, step=0.1)

    with col3:
        st.subheader("Menstrual & symptoms")
        cycle_irreg = st.radio("Menstrual cycle", ["Regular", "Irregular"])
        cycle_code = 4 if cycle_irreg == "Irregular" else 2
        cycle_len = st.number_input(
            "Cycle length code (2–12 per dataset encoding)",
            2, 12, 5,
            help="This dataset encodes cycle length as a category, not actual days. "
                 "Use Cycle(R/I) for the clinical criterion."
        )
        mfg_score = st.number_input(
            "mFG score (optional — leave 0 if not assessed)",
            0, 36, 0,
            help="Modified Ferriman-Gallwey score for hirsutism. "
                 "If 0, binary hair growth proxy will be used."
        )
        hair_growth = st.checkbox("Hirsutism / hair growth", value=False)
        skin_dark = st.checkbox("Acanthosis nigricans (skin darkening)", value=False)
        weight_gain = st.checkbox("Unexplained weight gain", value=False)
        pimples = st.checkbox("Persistent acne", value=False)
        hair_loss = st.checkbox("Female pattern hair loss", value=False)
        fast_food = st.checkbox("Fast food ≥3×/week", value=False)
        exercise = st.checkbox("Regular exercise (≥150 min/week)", value=False)

    ultrasound_data = {}
    if tier == 2:
        st.subheader("Ultrasound findings")
        ucol1, ucol2 = st.columns(2)
        with ucol1:
            fol_r = st.number_input("Follicle count — right ovary", 0, 40, 5)
            avg_f_r = st.number_input("Avg follicle size R (mm)", 0.0, 30.0, 8.0, step=0.5)
        with ucol2:
            fol_l = st.number_input("Follicle count — left ovary", 0, 40, 5)
            avg_f_l = st.number_input("Avg follicle size L (mm)", 0.0, 30.0, 8.0, step=0.5)
        endo_thick = st.number_input("Endometrial thickness (mm)", 0.0, 20.0, 8.0, step=0.5)
        st.caption(
            "Ovarian volume not captured in training dataset. "
            "The 2023 alternative threshold (>10 cm³) cannot be applied directly."
        )
        ultrasound_data = {
            'Follicle No. (R)': fol_r,
            'Follicle No. (L)': fol_l,
            'Avg. F size (R) (mm)': avg_f_r,
            'Avg. F size (L) (mm)': avg_f_l,
            'Endometrium (mm)': endo_thick,
        }

    patient = {
        'Age (yrs)': age,
        'BMI': bmi,
        'LH(mIU/mL)': min(lh, 100),  # cap per EDA finding
        'FSH(mIU/mL)': fsh,
        'FSH/LH': fsh_lh,
        'AMH(ng/mL)': amh,
        'TSH (mIU/L)': tsh,
        'Cycle(R/I)': cycle_code,
        'Cycle length(days)': cycle_len,
        'hair growth(Y/N)': int(hair_growth),
        'Skin darkening (Y/N)': int(skin_dark),
        'Weight gain(Y/N)': int(weight_gain),
        'Pimples(Y/N)': int(pimples),
        'Hair loss(Y/N)': int(hair_loss),
        'Fast food (Y/N)': int(fast_food),
        'Reg.Exercise(Y/N)': int(exercise),
        **ultrasound_data,
    }

    mfg = mfg_score if mfg_score > 0 else None
    return patient, mfg


# ── Rotterdam display ─────────────────────────────────────────────────────────
def render_rotterdam(result: RotterdamResult):
    st.subheader("Rotterdam criteria assessment")
    st.caption(f"Guideline version: {result.guideline_version}")

    def criterion_card(cr, label):
        if cr.met:
            color = "🟢"
            status = "Met"
        elif cr.partial:
            color = "🟡"
            status = "Partially met / borderline"
        else:
            color = "🔴"
            status = "Not met"

        with st.expander(f"{color} Criterion {label}: {cr.name} — {status}"):
            for e in cr.evidence:
                st.markdown(f"- {e}")
            if cr.caveat:
                st.warning(f"**Dataset caveat:** {cr.caveat}")
            st.caption(f"Threshold used: {cr.threshold_used}")

    criterion_card(result.criterion1, "1")
    criterion_card(result.criterion2, "2")
    criterion_card(result.criterion3, "3")

    met = result.criteria_met
    if result.diagnosis_supported:
        st.success(
            f"✅ {met}/3 Rotterdam criteria met — PCOS diagnosis supported. "
            f"Confirm after excluding thyroid disorders, hyperprolactinaemia, "
            f"and congenital adrenal hyperplasia."
        )
    elif met == 1:
        st.warning(
            f"⚠️ {met}/3 Rotterdam criteria met — insufficient for PCOS diagnosis. "
            f"Consider alternative diagnoses or repeat investigation in 3 months."
        )
    else:
        st.error(
            f"❌ {met}/3 Rotterdam criteria met — PCOS diagnosis not supported "
            f"by available data."
        )

    if result.dataset_caveats:
        with st.expander("View all dataset caveats"):
            for c in result.dataset_caveats:
                st.markdown(f"- {c}")


# ── ML output ─────────────────────────────────────────────────────────────────
def render_ml_output(patient: dict, tier: int, models: dict):
    st.subheader("Model probability output")

    model_key = 'tier1_lr' if tier == 1 else 'tier2_rf'
    feat_key = 'tier1_features' if tier == 1 else 'tier2_features'

    if model_key not in models:
        st.warning("Models not yet trained. Run train.py first.")
        return None

    pipeline = models[model_key]
    features = models[feat_key]

    row = pd.DataFrame([patient])[
        [f for f in features if f in pd.DataFrame([patient]).columns]
    ]
    row = row.reindex(columns=features)

    prob = pipeline.predict_proba(row)[0][1]

    col1, col2, col3 = st.columns(3)
    col1.metric(f"PCOS probability (Tier {tier})", f"{prob:.0%}")
    col2.metric("Model", "Logistic Regression" if tier == 1 else "Random Forest")
    if 'metrics' in models:
        auc = models['metrics'].get(f'tier{tier}', {}).get('roc_auc', None)
        if auc:
            col3.metric("Model AUROC (5-fold CV)", f"{auc:.3f}")

    st.progress(float(prob))

    if prob >= 0.7:
        st.error(
            f"**High PCOS likelihood ({prob:.0%})** — recommend specialist referral, "
            f"full androgen panel, and transvaginal ultrasound if not already done."
        )
    elif prob >= 0.4:
        st.warning(
            f"**Moderate PCOS likelihood ({prob:.0%})** — borderline. "
            f"Repeat hormonal panel on day 3 of cycle. Consider ultrasound."
        )
    else:
        st.info(
            f"**Low PCOS likelihood ({prob:.0%})** — consider alternative diagnoses: "
            f"thyroid disorder, hyperprolactinaemia, endometriosis."
        )

    return prob


# ── Differential output ───────────────────────────────────────────────────────
def render_differential(patient: dict, models: dict):
    st.subheader("Differential diagnosis")

    if 'diff_xgb' not in models:
        st.warning("Differential model not available.")
        return

    pipeline = models['diff_xgb']
    features = models['diff_features']

    row = pd.DataFrame([patient]).reindex(columns=features)
    probs = pipeline.predict_proba(row)[0]
    labels = ['Neither / other', 'PCOS', 'Endometriosis']

    col1, col2, col3 = st.columns(3)
    metrics_cols = [col1, col2, col3]
    for i, (label, prob) in enumerate(zip(labels, probs)):
        metrics_cols[i].metric(label, f"{prob:.0%}")

    st.caption("Based on joint classifier trained on PCOS + endometriosis datasets")

    # Distinguishing feature table
    st.markdown("**Key distinguishing features — PCOS vs endometriosis**")
    diff_df = pd.DataFrame([
        ("Follicle count", "Elevated (mean 10–11/ovary)", "Normal"),
        ("LH level", "Often elevated", "Usually normal"),
        ("AMH", "Elevated (mean 7.8 ng/mL)", "Normal or low"),
        ("Chronic pelvic pain", "Mild or absent", "Severe (defining feature)"),
        ("Hyperandrogenism signs", "Present (hirsutism, acne)", "Usually absent"),
        ("Menstrual irregularity", "Common (53% in dataset)", "Present but different pattern"),
    ], columns=["Feature", "PCOS pattern", "Endometriosis pattern"])
    st.dataframe(diff_df, use_container_width=True, hide_index=True)

    # Recommended next steps
    st.markdown("**Recommended investigations**")
    inv_col1, inv_col2 = st.columns(2)
    with inv_col1:
        st.markdown("""
- Free androgen index / testosterone
- Day 3 LH, FSH, estradiol
- Fasting glucose + insulin (HOMA-IR)
- Thyroid panel (TSH, T4) — rule out
- Prolactin — rule out hyperprolactinaemia
""")
    with inv_col2:
        st.markdown("""
- Transvaginal ultrasound (follicle count + ovarian volume)
- mFG score assessment (hirsutism grading)
- Lipid panel (metabolic risk)
- If endo suspected: CA-125, diagnostic laparoscopy
""")


# ── Phenotype clustering ───────────────────────────────────────────────────────
def render_phenotype(result: RotterdamResult):
    """Map patient to one of the four Rotterdam phenotypes (A–D)."""
    st.subheader("PCOS phenotype")

    c1 = result.criterion1.met
    c2 = result.criterion2.met
    c3 = result.criterion3.met

    if c1 and c2 and c3:
        phenotype = "A"
        desc = "Classic PCOS — oligo-anovulation + hyperandrogenism + polycystic ovaries. Most severe metabolic risk."
    elif c1 and c2 and not c3:
        phenotype = "B"
        desc = "Classic PCOS without polycystic ovaries — oligo-anovulation + hyperandrogenism. High metabolic risk."
    elif c2 and c3 and not c1:
        phenotype = "C"
        desc = "Ovulatory PCOS — hyperandrogenism + polycystic ovaries, regular cycles. Milder metabolic profile."
    elif c1 and c3 and not c2:
        phenotype = "D"
        desc = "Non-androgenic PCOS — oligo-anovulation + polycystic ovaries, no hyperandrogenism. Lowest metabolic risk."
    else:
        phenotype = "?"
        desc = "Insufficient criteria for phenotype classification. ≥2 criteria required."

    if phenotype != "?":
        st.success(f"**Phenotype {phenotype}** — {desc}")
        st.caption(
            "Phenotype classification guides prognosis and management: "
            "Phenotypes A/B carry highest cardiovascular and metabolic risk; "
            "C/D have more favourable long-term profiles."
        )
    else:
        st.warning(desc)


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    tier, ethnicity, use_2023 = render_sidebar()

    models = load_models()

    if not models:
        st.warning(
            "No trained models found. Please run `python train.py` first. "
            "The Rotterdam criteria assessment will still work without models."
        )

    patient, mfg = render_input_form(tier)

    st.divider()

    if st.button("Run assessment", type="primary", use_container_width=True):
        rotterdam_result = apply_rotterdam(
            patient, mfg_score=mfg, ethnicity=ethnicity,
            apply_2023_guidelines=use_2023
        )

        tab1, tab2, tab3, tab4 = st.tabs([
            "Rotterdam criteria",
            "ML probability",
            "Differential diagnosis",
            "Phenotype",
        ])

        with tab1:
            render_rotterdam(rotterdam_result)

        with tab2:
            render_ml_output(patient, tier, models)

        with tab3:
            render_differential(patient, models)

        with tab4:
            render_phenotype(rotterdam_result)


if __name__ == '__main__':
    main()


Overwriting app.py


# 4. Run server

In [ ]:
""" Run this first before the website to force clear the old server """
import subprocess

# Kill any old background servers just in case
!fuser -k 8501/tcp


subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false'])

print("Streamlit server started in the background!")

Streamlit server started in the background!


# 5. Run the website

In [ ]:
"""
Enter the password provided below into the website link.
Hit REFRESH until website works.
"""
!npm install -g localtunnel

# Fetch the correct IP address for the Localtunnel password
import urllib
print("Password/Tunnel Password:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip('\n'))

# Start the tunnel
!npx localtunnel --port 8501


⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
changed 22 packages in 1s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸Password/Tunnel Password: 34.158.70.110
⠙⠹⠸⠼your url is: https://fresh-jeans-eat.loca.lt
^C
